# 🌍 → 🌕 / 🔴  Earth Analog Finder
A LightGBM model that predicts whether a place on Earth is a **Moon analog**, a **Mars analog**, or **neither**.

Space agencies train astronauts and test rovers at places on Earth that resemble the Moon or Mars, such as Meteor Crater and the Atacama Desert. This notebook learns what those places have in common from NASA data, then scores **any** coordinate, for example `Moon 0.62 · Mars 0.25 · Neither 0.13`.

| Section | Content |
|---|---|
| 1 | Setup |
| 2 | Labelled places (`data/sites.csv`) |
| 3 | Data: NASA imagery, terrain, climate, surface temperature, PDS spectra |
| 4 | Features |
| 5 | Look at the data |
| 6 | Model (LightGBM) |
| 7 | Evaluation on unseen places |
| 8 | Feature importance |
| 9 | Predict new places |
| 10 | Export the model for the website (`webapp/`) |

---
## 1. Setup
**Libraries:** `requests` (web APIs), `numpy` and `pandas`, `Pillow` (images), `lightgbm` (model), `scikit-learn` (evaluation), `matplotlib` (charts).

Every download is cached in `data/cache/`, so re-runs are fast.

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from PIL import Image
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

In [ ]:
PATCH_HALF_DEG  = 0.10     # area around each place: +/-0.1 deg (~ +/-11 km)
PATCH_PX        = 48       # image is 48 x 48 pixels (~460 m / pixel)
PIXELS_PER_SITE = 200      # training pixels sampled per place
SEED            = 42

CLASSES = ["Moon", "Mars", "Neither"]                       # class 0, 1, 2
COLORS  = {"Moon": "#2a78d6", "Mars": "#eb6834", "Neither": "#1baf7a"}   # fixed colour per class

DATA = Path("..") / "data"; DATA.mkdir(exist_ok=True)       # the notebook lives in model/, the data one level up
CACHE = DATA / "cache"; CACHE.mkdir(exist_ok=True)
HEADERS = {"User-Agent": "earth-analog-finder/0.1"}
rng = np.random.default_rng(SEED)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#888", "axes.labelcolor": "#333", "xtick.color": "#555",
                     "ytick.color": "#555", "axes.grid": True, "grid.color": "#eee"})